# Spark local setup (MinIO + Delta)
Reusable notebook to initialize a local SparkSession configured for Delta Lake on MinIO.

**Usage**: Run this notebook first, then other notebooks can rely on the `spark`, `bronze_path`, `silver_path`, `gold_path` variables.

In [14]:
import os
from pyspark.sql import SparkSession

# ---- Environment (override via env vars if needed) ----
S3A_ENDPOINT = os.getenv('S3A_ENDPOINT', 'http://localhost:9000')
S3A_ACCESS   = os.getenv('S3A_ACCESS_KEY', 'minio')
S3A_SECRET   = os.getenv('S3A_SECRET_KEY', 'minio123')
DELTA_LOG    = 'org.apache.spark.sql.delta.storage.S3SingleDriverLogStore'
BUCKET       = os.getenv('MINIO_BUCKET', 'datalake')

# ---- Spark config ----
SPARK_VERSION = "3.5.1"
SCALA_BINARY  = "2.12"
DELTA_VERSION = "3.2.0"
HADOOP_AWS_VERSION = "3.3.4"
AWS_SDK_BUNDLE_VERSION = "1.12.767"

packages = ",".join([
    f"io.delta:delta-spark_{SCALA_BINARY}:{DELTA_VERSION}",
    f"org.apache.hadoop:hadoop-aws:{HADOOP_AWS_VERSION}",
    f"com.amazonaws:aws-java-sdk-bundle:{AWS_SDK_BUNDLE_VERSION}",
])

# ---- Build Spark ----
spark = (
    SparkSession.builder
    .appName('nb-local')
    .master('local[*]')
    .config('spark.jars.packages', packages)
    .config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', S3A_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', S3A_ACCESS)
    .config('spark.hadoop.fs.s3a.secret.key', S3A_SECRET)
    .config('spark.hadoop.fs.s3a.path.style.access','true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled','false')
    .config('spark.hadoop.fs.s3a.impl','org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.delta.logStore.class', DELTA_LOG)
    .getOrCreate()
)

# ---- Common paths ----
bronze_path = f's3a://{BUCKET}/bronze/reddit_posts'
silver_path = f's3a://{BUCKET}/silver/reddit_posts'
gold_path   = f's3a://{BUCKET}/gold/reddit_posts'
bronze_path, silver_path, gold_path

('s3a://datalake/bronze/reddit_posts',
 's3a://datalake/silver/reddit_posts',
 's3a://datalake/gold/reddit_posts')

In [15]:
# Quick sanity checks (non-fatal)
from pyspark.sql import functions as F

def try_head(path: str, n: int = 5):
    try:
        df = spark.read.format("delta").load(path)

        print(f"Loaded: {path} | count≈", df.limit(1_000_000).count())

        candidates = ["ingest_ts", "created_utc", "event_time", "dt"]
        sort_col = next((c for c in candidates if c in df.columns), None)

        if sort_col:
            df.orderBy(F.desc(sort_col)).show(n, truncate=False)
        else:
            df.show(n, truncate=False)
    except Exception as e:
        print(f"Cannot read {path}:", e)

try_head(bronze_path)
try_head(silver_path)
try_head(gold_path)

Loaded: s3a://datalake/bronze/reddit_posts | count≈ 60
+-------+----------+-----------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------